# AXI test 30, follow-up: how XLA lays out the cache-shaped arrays on a TPU (answers a question on jax-ml/jax Discussion #42904)

Runtime > Change runtime type > TPU v6e-1 (or v5e-1 if v6e is not offered; the device name is printed first, so say which chip you ran on). No key; a few minutes. No model is run. It prints, for each token length of Test 30 and for the next multiple of 128 above it: the bytes the chip reports in use after the array is put on it, `Compiled.memory_analysis()` for an identity function on the array, and the layout of the array in the compiled HLO. Pre-registered page: `docs/TPU_SESSIONS_TEST.md` (follow-up section).

In [ ]:
import gc, json, re
import jax, jax.numpy as jnp
dev = jax.devices()[0]
print("jax", jax.__version__, "|", dev, "|", getattr(dev, "device_kind", "?"))
LENGTHS = [2824, 2944, 50000, 50048, 100000, 100096, 250000, 250112, 476835, 476928, 949591, 949632]

def in_use():
    try:
        return (dev.memory_stats() or {}).get("bytes_in_use")
    except Exception:
        return None

def safe(f):
    try:
        return f()
    except Exception as e:
        return "ERR " + type(e).__name__ + ": " + str(e)[:120]

rows = []
first_hlo = {}
for t in LENGTHS:
    nominal = 2 * 1 * 2 * t * 64 * 2
    gc.collect(); b0 = in_use()
    arr = jnp.zeros((2, 1, 2, t, 64), dtype=jnp.bfloat16); arr.block_until_ready()
    b1 = in_use()
    low = jax.jit(lambda x: x).lower(arr)
    comp = low.compile()
    ma = safe(lambda: comp.memory_analysis())
    txt = safe(lambda: comp.as_text())
    lay = re.findall(r"bf16\[[0-9,]+\]\{[^}]*\}", txt) if isinstance(txt, str) else None
    row = {"tokens": t, "nominal_bytes": nominal,
           "bytes_in_use_delta": (b1 - b0) if (b0 is not None and b1 is not None) else None,
           "on_device_size_in_bytes": safe(lambda: arr.on_device_size_in_bytes()),
           "layout_in_compiled_hlo": lay[:2] if lay else lay}
    for k in ("argument_size_in_bytes", "output_size_in_bytes", "temp_size_in_bytes", "alias_size_in_bytes"):
        row[k] = safe(lambda: getattr(ma, k))
    rows.append(row); print(json.dumps(row))
    if t in (2824, 2944) and isinstance(txt, str):
        first_hlo[t] = "\n".join(txt.splitlines()[:25])
    del arr, low, comp; gc.collect()

per_array = []
for t in (2824, 2944):
    nominal = 2 * 1 * 2 * t * 64 * 2
    gc.collect(); b0 = in_use()
    arrs = [jnp.zeros((2, 1, 2, t, 64), dtype=jnp.bfloat16) for _ in range(24)]
    [a.block_until_ready() for a in arrs]
    b1 = in_use()
    if b0 is not None and b1 is not None:
        d = (b1 - b0) / 24
        per_array.append({"tokens": t, "arrays": 24, "delta_per_array": d, "nominal": nominal, "extra_per_array": d - nominal})
        print(json.dumps(per_array[-1]))
    del arrs; gc.collect()

for t, h in first_hlo.items():
    print("\n==== compiled HLO, tokens =", t, "====\n" + h)

out = {"jax": jax.__version__, "device": str(dev), "rows": rows, "per_array": per_array}
json.dump(out, open("/content/tpu_layout_check.json", "w"), indent=1)
print("\nsaved /content/tpu_layout_check.json")